# Batch A — Validation-calibrated audit scoring

Companion notebook for `src/audit_scoring`.

This notebook **imports package functions**; it does not re-implement scoring logic.
Diagnostic scores do **not** predict future performance and do **not** certify deployment readiness.

Run the CLI pipeline first to freeze reference files, then execute this notebook top-to-bottom.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

repository_root = Path(r"C:\git\tesis\DT_ABLATION_RISK").resolve()
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

diagnostic_pkl_root = Path(r"C:\git\tesis\results\ablation_ESWA_all_models")
metrics_dir = repository_root / "results" / "mendeley" / "dt_metrics"
model_assessment_path = Path(
    r"C:\git\DT-DIO-ETSII-UPM\src\portfolio_optimization\results\ESWA_wHoldout\model_assessment.pkl"
)
output_dir = metrics_dir / "audit_screening"
primary_cost_bps = 5
selected_model_id = None  # filled after loading scored table if None
selected_model_family = "efficient_risk_TI_True"
force_recompute = False
export_outputs = False  # notebook does not overwrite pipeline outputs by default

from src.audit_scoring.config import AuditConfig
from src.audit_scoring.pipeline import run_batch_a, inspect_inventory
from src.audit_scoring.io import read_table

print("repository_root:", repository_root)
print("output_dir:", output_dir)

## Optional recompute

Frozen Batch A artefacts should already exist under `audit_screening`.
Set `force_recompute=True` only when you intentionally want to regenerate them.

In [ ]:
config = AuditConfig(
    repository_root=repository_root,
    diagnostic_pkl_root=diagnostic_pkl_root,
    metrics_dir=metrics_dir,
    model_assessment_pickle=model_assessment_path,
    output_dir=output_dir,
    primary_cost_bps=primary_cost_bps,
    force=force_recompute,
)

required = [
    output_dir / "audit_pkl_inventory.csv",
    output_dir / "audit_model_diagnostics_scored.csv",
    output_dir / f"audit_validation_decisions_primary_{primary_cost_bps}bps.csv",
    output_dir / "audit_validation_thresholds.json",
]
missing = [p for p in required if not p.exists()]
if force_recompute or missing:
    print("Running Batch A pipeline…", "missing=" , [str(p.name) for p in missing])
    summary = run_batch_a(config)
    print(summary)
else:
    print("Using frozen Batch A outputs from", output_dir)

## 2. Data and mapping inspection

In [ ]:
inventory = read_table(output_dir / "audit_pkl_inventory.csv")
mapping = read_table(output_dir / "audit_model_mapping.csv")
raw = read_table(output_dir / "audit_model_diagnostics_raw.csv")
model_scored = read_table(output_dir / "audit_model_diagnostics_scored.csv")
family_raw = read_table(output_dir / "audit_family_diagnostics_raw.csv")
family_scored = read_table(output_dir / "audit_family_diagnostics_scored.csv")
primary = read_table(output_dir / f"audit_validation_decisions_primary_{primary_cost_bps}bps.csv")

print("PKLs inventoried:", len(inventory))
print(mapping["mapping_status"].value_counts(dropna=False))
print("\nModels by family:")
display(raw.groupby(["model_family", "technical_indicators"]).size().rename("n").reset_index())
print("TI coverage:")
display(raw["technical_indicators"].value_counts(dropna=False))

## 3. Score / inspect one selected individual model

Uses **frozen** model-level reference distributions from the Batch A package outputs.

In [ ]:
if selected_model_id is None and not model_scored.empty:
    selected_model_id = str(model_scored["model_id"].iloc[0])

row_raw = raw[raw["model_id"].astype(str) == str(selected_model_id)]
row_scored = model_scored[model_scored["model_id"].astype(str) == str(selected_model_id)]
print("Selected model:", selected_model_id)
display(row_raw.T)
display(row_scored.T)

fam = str(row_scored["model_family"].iloc[0]) if not row_scored.empty else selected_model_family
within = model_scored[model_scored["model_family"] == fam].sort_values("diagnostic_score_model", ascending=False)
print("Within-family position:")
display(within[["model_id", "diagnostic_score_model", "contribution_score_model", "necessity_score_model", "structure_score_model"]])
print("Global rank by diagnostic_score_model:")
glob = model_scored.sort_values("diagnostic_score_model", ascending=False).reset_index(drop=True)
display(glob.loc[glob["model_id"].astype(str) == str(selected_model_id), ["model_id", "model_family", "diagnostic_score_model"]])

## 4. Inspect one selected family

In [ ]:
fam_row = family_scored[family_scored["model_family"] == selected_model_family]
fam_raw_row = family_raw[family_raw["model_family"] == selected_model_family]
dec = primary[primary["model_family"] == selected_model_family]
print("Selected family:", selected_model_family)
display(fam_raw_row.T)
display(fam_row.T)
display(dec.T)
if not dec.empty:
    print("Diagnostic band:", dec["diagnostic_band"].iloc[0], "|", dec["diagnostic_band_reason"].iloc[0])
    print("Performance band:", dec["performance_band"].iloc[0])
    print("Recommendation:", dec["operational_recommendation"].iloc[0])
    print("Decision reason:", dec["decision_reason"].iloc[0])

## 5. Full 18-family primary screening table (5 bps)

In [ ]:
order = {"ADVANCE": 0, "MONITOR": 1, "INVESTIGATE": 2, "REJECT": 3}
table = primary.copy()
table["decision_rank"] = table["operational_recommendation"].map(order)
table = table.sort_values(
    ["decision_rank", "diagnostic_score", "median_sharpe"],
    ascending=[True, False, False],
)
cols = [
    "model_family", "operational_recommendation", "diagnostic_band", "performance_band",
    "diagnostic_score", "contribution_score", "necessity_score", "structure_score",
    "median_sharpe", "median_max_drawdown", "median_annualized_turnover",
    "high_drawdown_flag", "high_turnover_flag", "model_coverage_ratio", "decision_reason",
]
cols = [c for c in cols if c in table.columns]
display(table[cols])

## 6. Diagnostic visualizations

The diagnostic-score versus validation-Sharpe scatter is **descriptive**, not predictive.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col, title in zip(
    axes.ravel(),
    ["contribution_score", "necessity_score", "structure_score", "diagnostic_score"],
    ["Contribution", "Necessity", "Structure", "Aggregate diagnostic"],
):
    s = primary.sort_values(col)
    ax.barh(s["model_family"], s[col])
    ax.set_title(title + " by family")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(primary["diagnostic_score"], primary["median_sharpe"])
for _, r in primary.iterrows():
    ax.annotate(r["model_family"], (r["diagnostic_score"], r["median_sharpe"]), fontsize=7)
ax.set_xlabel("Diagnostic score (validation)")
ax.set_ylabel("Validation median Sharpe (5 bps)")
ax.set_title("Descriptive joint view (not a predictive model)")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(family_raw["model_family"], family_raw["contribution_iqr_summary"])
ax.tick_params(axis="x", rotation=90)
ax.set_title("Within-family contribution IQR summary")
plt.tight_layout()
plt.show()

if not model_scored.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    model_scored.boxplot(column="diagnostic_score_model", by="model_family", ax=ax, rot=90)
    ax.set_title("Model-level diagnostic score by family")
    plt.suptitle("")
    plt.tight_layout()
    plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
for rec, sub in primary.groupby("operational_recommendation"):
    ax.scatter(sub["diagnostic_score"], sub["median_sharpe"], label=rec)
ax.legend()
ax.set_xlabel("Diagnostic score")
ax.set_ylabel("Validation Sharpe")
ax.set_title("Decision-matrix placement (descriptive)")
plt.show()

## 7. Export controls

By default this notebook does **not** overwrite pipeline outputs.

In [ ]:
if export_outputs:
    export_path = output_dir / "notebook_exports"
    export_path.mkdir(parents=True, exist_ok=True)
    table[cols].to_csv(export_path / "primary_screening_table_from_notebook.csv", index=False)
    print("Wrote optional notebook export to", export_path)
else:
    print("export_outputs=False — no files written.")

## 8. Final summary

In [ ]:
print("Selected model:", selected_model_id)
print("Selected family:", selected_model_family)
print("Evidence available: contribution / necessity / structure from diagnostic PKLs + validation family performance.")
print("Missing validity checks: alternative baselines, parameter randomization, input permutation, feature-deletion curves, layer conductance (as scored component).")
if not dec.empty:
    print("Operational meaning:", dec["operational_recommendation"].iloc[0], "—", dec["decision_reason"].iloc[0])
print("Explicit statement: the diagnostic score does not predict future performance.")